# plan-a-closure Tier 0 with the long trial, on Kaggle's two T4s (engineering shakedown)

This runs the one-command Tier 0 pilot with its long trial (`closure tier0 --trial-profile long`) on each GPU of this
Kaggle session, at the same time, and then reads the two trials as one campaign (`closure pool`). Each trial runs about
9 hours, the longest that fits Kaggle's 12-hour limit: 204 null blocks, 12 hidden blocks for every class and intensity,
144 calibration blocks interleaved through the run. Two trials give 408 null blocks and about 200 hidden blocks at 5% of
capacity or more, enough to read the pre-registered decision rules with room to spare (`docs/PREREGISTRATION.md`,
amendment v0.6).

**What it is.** A shakedown on a non-target device, in a container, with NVML as the only power source.

**What it is not.** The registered trial. There is no independent power meter, so assumption A6 does not hold, and the
registered numbers come from Tier 1 on instrumented H100/H200 hardware. Every run is published whatever it shows
(`docs/DEVIATIONS.md`).

**How to run it** (about 10.5 hours, unattended; nothing waits for you)

1. Your Kaggle account must be phone-verified (Settings > Phone verification): Kaggle needs it for GPUs and internet.
2. In the notebook's right-hand panel: Accelerator **GPU T4 x2** (not P100: it has no NVML energy counter) and
   Internet **on**. Check that at least 11 GPU hours of this week's quota are left.
3. **Save Version** (top right) > **Save & Run All (Commit)** > Save. The committed run continues on its own after
   you close the tab. Then stop the interactive editing session (its power icon), so it does not use quota as well.
4. After about 10.5 hours, open the saved version (the notebook's Versions list) > **Output** and download everything
   in the run's folder: a `tier0_bundle.tar.gz` and `SUMMARY.md` for each GPU, `CAMPAIGN.md`, `pooled.json` and
   `drill.json`.

The private signing keys never reach the saved output: the runs happen outside `/kaggle/working`, and only those files
are copied there. Keep the notebook private until you publish the result.

In [ ]:
# 1. Setup: the GPUs, internet, the exact code. The runs happen under /tmp, outside /kaggle/working (which Kaggle saves
#    with the version), so only the deliverables are saved, never a private signing key.
import os, shutil, subprocess, time
REPO, REF = "https://github.com/AnonRish/plan-a-closure", "main"   # REF can be a tag or a commit, to pin the code
SRC, RUNS, SAVE = "/tmp/plan-a-closure", "/tmp/plan-a-closure-runs", "/kaggle/working"
T_START = time.time()
q = subprocess.run(["nvidia-smi", "--query-gpu=index,name,uuid,driver_version,memory.total,power.limit", "--format=csv,noheader"],
                   capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if q is None or q.returncode != 0 or not q.stdout.strip():
    raise SystemExit("No GPU: in the right-hand panel set Accelerator to GPU T4 x2, then Save & Run All again.")
print(q.stdout)
if "P100" in q.stdout:
    raise SystemExit("This is a P100, which has no NVML energy counter: choose GPU T4 x2, then Save & Run All again.")
N_GPU = len(q.stdout.strip().splitlines())
try:
    online = subprocess.run(["git", "ls-remote", "-q", REPO, "HEAD"], capture_output=True, timeout=60).returncode == 0
except subprocess.TimeoutExpired:
    online = False
if not online:
    raise SystemExit("No internet: turn Internet on in the right-hand panel (it needs a phone-verified account).")
if not os.path.isdir(f"{SRC}/.git"):
    !git clone -q --depth 1 {REPO} {SRC}
!git -C {SRC} fetch -q --depth 1 origin {REF} && git -C {SRC} checkout -q FETCH_HEAD
%cd /tmp/plan-a-closure
!pip install -q -e ".[test,hw]"
!pip install -q opentimestamps-client    # stamps each pre-registration on public calendars before any trial data exist
!python -c "import closure; print('plan-a-closure', closure.__version__)"
!git log -1 --format="code: commit %H (%cd)"
!nvidia-smi -pm 1 > /dev/null 2>&1 && echo "persistence mode on" || echo "persistence mode not available here (fine)"

In [ ]:
# 2. Preflight on every GPU: NVML, the energy counter, torch and CUDA, which physical GPU. FAIL lines here stop that
#    GPU's run in cell 3; WARN lines for fp8, clock sync and tcpdump are normal on Kaggle.
for g in range(min(2, N_GPU)):
    !closure preflight --gpu {g}

## The run (about 10 hours)

Cell 3 starts Tier 0 with the long trial on GPU 0, and on GPU 1 as soon as GPU 0's unit tests are done (they need the
CPU to themselves). The two trials are declared as one campaign before either starts and each draws its own seed, so
they follow different load sequences and different committed schedules. Tier 0 logs every step, sizes the unit of work
to the GPU, measures the hidden load on its own so that its labels mean what they say, locks the trial's clocks where
the container allows it (the log says whether it did), restores the GPU on exit, and writes `SUMMARY.md` and
`tier0_bundle.tar.gz`. Then the cell copies the deliverables into `/kaggle/working`, pools the two trials and runs A
Narrow Path's mock training run on GPU 0. If a run is still going 11 hours 20 minutes after the session started, it is
stopped cleanly so the summary and the bundle of every finished step are still saved before Kaggle's 12-hour limit.
If GPU 0 stops at its unit tests, the cell saves that log and starts both trials without them, so the session is not
lost; the output then says so and it belongs in the report.

In [ ]:
# 3. The runs: Tier 0 with the long trial on each GPU, the deliverables, the pooled reading, the mock training run.
import hashlib
from IPython.display import Markdown, display
STAMP = time.strftime("%Y%m%d-%H%M%S")
GPUS = list(range(min(2, N_GPU)))
CAMPAIGN = f"kaggle-{STAMP}"
DEADLINE = 11 * 3600 + 20 * 60                                      # seconds after the session start: stop cleanly
os.makedirs(RUNS, exist_ok=True)

def launch(i, gpu, skip_tests=False, tag=""):
    out = f"{RUNS}/tier0-{STAMP}-gpu{gpu}{tag}"
    cmd = ["timeout", "--signal=TERM", str(int(DEADLINE - (time.time() - T_START))), "closure", "tier0",
           "--confirm-hardware", "--gpu", str(gpu), "--trial-profile", "long", "--out", out]
    cmd += ["--campaign", f"{CAMPAIGN}:{i + 1}/{len(GPUS)}"] if len(GPUS) > 1 else []
    cmd += ["--skip-tests"] if skip_tests else []
    return {"gpu": gpu, "out": out, "seen": 0, "log": f"{out}.console.log",
            "proc": subprocess.Popen(cmd, stdout=open(f"{out}.console.log", "w"), stderr=subprocess.STDOUT,
                                     env=dict(os.environ, PYTHONUNBUFFERED="1"))}

def relay(job):                                                      # new console lines, so the version's log shows life
    lines = open(job["log"], errors="replace").read().splitlines()
    for line in lines[job["seen"]:]:
        print(f"[GPU {job['gpu']}] {line}", flush=True)
    job["seen"] = len(lines)
    return lines

def past_the_unit_tests(job):
    return any(line.startswith("[2]") for line in relay(job))

jobs, first = [launch(0, GPUS[0])], None                             # the unit tests run once, with GPU 0, alone on the CPU
while jobs[0]["proc"].poll() is None and not past_the_unit_tests(jobs[0]):
    time.sleep(20)
if not past_the_unit_tests(jobs[0]):
    first = jobs[0]                                                  # it stopped at its first step
    print("GPU 0 stopped before its second step: the unit tests failed on this machine, or the run could not start. "
          "Its log is saved with the output (gpu0-first-attempt-*). Starting again without the unit tests, so that "
          "the session is not lost: say so when you report these runs.", flush=True)
    jobs = [launch(0, GPUS[0], skip_tests=True, tag="-b")]
for i in range(1, len(GPUS)):
    jobs.append(launch(i, GPUS[i], skip_tests=True))
while any(j["proc"].poll() is None for j in jobs):
    for j in jobs:
        relay(j)
    time.sleep(120)
for j in jobs:
    relay(j)
    print(f"[GPU {j['gpu']}] tier0 exit {j['proc'].returncode}", flush=True)

dest = os.path.join(SAVE, f"tier0-{STAMP}")
os.makedirs(dest, exist_ok=True)
for j in jobs:                                                       # first of all, per GPU: the bundle and the summary
    for name in ("tier0_bundle.tar.gz", "SUMMARY.md"):
        if os.path.exists(f"{j['out']}/{name}"):
            shutil.copy2(f"{j['out']}/{name}", f"{dest}/gpu{j['gpu']}-{name}")
if first:
    for path, name in ((f"{first['out']}/logs/01-unit-tests.log", "unit-tests.log"), (f"{first['out']}/SUMMARY.md", "SUMMARY.md"),
                       (first["log"], "console.log")):
        if os.path.exists(path):
            shutil.copy2(path, f"{dest}/gpu0-first-attempt-{name}")
trials = [f"{j['out']}/T1-long" for j in jobs]
if len(jobs) > 1 and all(os.path.exists(f"{t}/results.json") for t in trials):
    pooled = subprocess.run(["closure", "pool", "--runs", *trials, "--out", f"{dest}/pooled.json"], capture_output=True, text=True)
    text = pooled.stdout + pooled.stderr
    open(f"{dest}/CAMPAIGN.md", "w").write(f"# Campaign {CAMPAIGN}: {len(jobs)} trials read as one\n\n{text}")
    print(text)
elif len(jobs) > 1:
    print("No pooled reading: a campaign is read whole or not at all, and not every trial finished. Each run's own "
          "SUMMARY.md stands.")
OUT = jobs[0]["out"]
if os.path.exists(f"{OUT}/probe.json") and time.time() - T_START < 11.5 * 3600:
    !closure treaty mock-run --device cuda:0 --probe {OUT}/probe.json --budget-flop 1e16 --dim 4096 --layers 8 --batch 2048 --out {OUT}/mock-run
if os.path.exists(f"{OUT}/mock-run/drill.json"):
    shutil.copy2(f"{OUT}/mock-run/drill.json", dest)
print("saved with this version's output:", dest, sorted(os.listdir(dest)))
for name in sorted(os.listdir(dest)):
    if name.endswith(".tar.gz"):
        print(name, "sha256:", hashlib.sha256(open(f"{dest}/{name}", "rb").read()).hexdigest())
for name in sorted(os.listdir(dest)):
    if name.endswith(".md"):
        display(Markdown(open(f"{dest}/{name}", encoding="utf-8").read()))

## After the run

* Download the run's folder from the saved version's Output. On Windows,
  `Get-FileHash $HOME\Downloads\gpu0-tier0_bundle.tar.gz` should match the sha256 printed above.
* `CAMPAIGN.md` reads the pre-registered decision rules on the two trials together (`closure pool`); each
  `SUMMARY.md` reads them on its own trial. They are shakedown readings: NVML only, so no Plan A claim.
* Each pre-registration was stamped with OpenTimestamps before its trial began (`T1-long/prereg.json.ots` in the
  bundle). A few hours later `ots upgrade prereg.json.ots` and `ots verify prereg.json.ots` tie it to a Bitcoin block.
* Publish the result whatever it shows, labelled as a shakedown: a new section in `docs/DEVIATIONS.md` (date, GPU,
  "Kaggle container", no independent meter, the bundles' SHA-256) and the bundles under `results/`.
* The next step is the registered run: Tier 0 and then Tier 1 on instrumented H100/H200 hardware
  (`docs/HARDWARE_CONFIG.md`).